# Zone C Vibration Detection & Root Cause Diagnostics (ISO 10816-7)
### Industrial Pump Condition Monitoring & 3-Way Fault Classification using Apache Spark

This notebook provides an end-to-end industrial analytics pipeline:
1. **Standards-Based Thresholds**: References ISO 10816-7 and Europump Vibration Guidelines (2013) to define Zone C boundaries.
2. **PySpark Data Cleaning**: Ingests telemetry datasets, handles missing values (~8% nulls) via window forward/backward fill.
3. **High Frequency Vibration with High Temperature**: Simulates and analyzes bearing defects with thermal anomalies.
4. **3-Way Fault Diagnosis**: Automatically attributes Zone C crossings to:
   - **Imbalance** (1x dominant, high radial vibration, normal temperature)
   - **Misalignment** (Coordinated multi-bearing vibration, high axial component, normal temperature)
   - **High Frequency Vibration with High Temperature** (High orders/harmonics, localized bearing defect, temperature > 65°C)

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure PySpark environment
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName('PumpDiagnosticsNotebook') \
    .master('local[2]') \
    .config('spark.driver.host', '127.0.0.1') \
    .config('spark.driver.bindAddress', '127.0.0.1') \
    .config('spark.sql.shuffle.partitions', '2') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark initialized: version {spark.version}')

## 1. ISO 10816-7 Vibration Limits & Zone C Standards
According to **BS ISO 10816-7 / Europump Guidelines (July 2013, Page 9)**, vibration velocity limits (mm/s RMS) on non-rotating parts are:
| Category & Power Rating | Zone A Limit | Zone B Upper Limit (Zone C Entry) | Zone C Upper Limit (Zone D Entry) | Max Alarm Limit |
|---|---|---|---|---|
| **Category 1 (<= 200 kW)** | 2.5 mm/s | **4.0 mm/s** | 6.6 mm/s | 5.0 mm/s |
| **Category 1 (> 200 kW)** | 3.5 mm/s | **5.0 mm/s** | 7.6 mm/s | 6.3 mm/s |
| **Category 2 (<= 200 kW)** | 3.2 mm/s | **5.1 mm/s** | 8.5 mm/s | 6.4 mm/s |
| **Category 2 (> 200 kW)** | 4.2 mm/s | **6.1 mm/s** | 9.5 mm/s | 7.6 mm/s |
| **Project Severe Threshold** | 3.5 mm/s | 5.0 mm/s | **7.0 mm/s** | 6.3 mm/s |

In [2]:
from spark_data_cleaning import clean_vibration_telemetry

# Clean datasets with PySpark
cleaned_misalign_df = clean_vibration_telemetry(
    spark, 
    'high_reading_pump_motor_vibration_dataset_2024_2025.csv',
    'cleaned_misalignment_dataset_2024_2025.csv'
)

cleaned_hf_ht_df = clean_vibration_telemetry(
    spark, 
    'pump_high_frequency_high_temperature_dataset_2024_2025.csv',
    'cleaned_high_freq_high_temp_dataset_2024_2025.csv'
)
print('Spark Cleaning complete with 0 missing values.')

In [3]:
from zone_c_fault_diagnostics import diagnose_telemetry_dataset, diagnose_frequency_spectrum

# Run diagnostics on Misalignment Dataset
res_misalign, counts_misalign = diagnose_telemetry_dataset(
    spark, 
    'cleaned_misalignment_dataset_2024_2025.csv',
    custom_threshold=7.0
)

# Run diagnostics on High Frequency + High Temp Dataset
res_hf_ht, counts_hf_ht = diagnose_telemetry_dataset(
    spark, 
    'cleaned_high_freq_high_temp_dataset_2024_2025.csv',
    custom_threshold=7.0
)

# Run diagnostics on Imbalance & High Frequency Frequency Spectra
spec_imb = diagnose_frequency_spectrum('pump_imbalance_frequency_spectrum.csv', threshold=7.0)
spec_hf = diagnose_frequency_spectrum('pump_high_frequency_spectrum.csv', threshold=7.0)

In [4]:
# Visual comparison of diagnosed causes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if counts_misalign:
    axes[0].pie(counts_misalign.values(), labels=counts_misalign.keys(), autopct='%1.1f%%', colors=['#ff7f0e', '#1f77b4'])
    axes[0].set_title('Misalignment Dataset: Zone C Breakdown')

if counts_hf_ht:
    axes[1].pie(counts_hf_ht.values(), labels=counts_hf_ht.keys(), autopct='%1.1f%%', colors=['#d62728'])
    axes[1].set_title('High Freq + High Temp Dataset: Zone C Breakdown')

plt.tight_layout()
plt.show()